# Sprint 4 — 14_final_validation.ipynb
## Rol: Final Validator + cierre del Experiment Tracker (PB-15)

### Objetivo
Evaluar el mejor modelo del Sprint 4 en el **test set final** (una sola vez), cuantificar la mejora vs baseline del Sprint 3, calcular intervalos de confianza y dejar persistido el modelo final.

### Decisiones de este template
- **Métrica principal por defecto**: `precision`.
- Debes definir manualmente `FINAL_MODEL_KEY` antes de tocar el test set.
- La evaluación final está protegida por `CONFIRM_FINAL_TEST_EVALUATION = False`.
- Este notebook también actualiza `models/experiments_log.csv`.

### Outputs esperados
- `models/final_model.pkl`
- `models/final_validation_metrics.csv`
- `models/baseline_vs_final_comparison.csv`
- `reports/figures/sprint4/final_confusion_matrix.png`
- `reports/figures/sprint4/final_bootstrap_intervals.png`
- `models/experiments_log.csv` actualizado

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
PROJECT_ROOT = Path(".").resolve().parent if Path.cwd().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "figures" / "sprint4"

TARGET_COL = "IsBadBuy"
PRIMARY_METRIC = "precision"

FINAL_MODEL_KEY = None
CONFIRM_FINAL_TEST_EVALUATION = False

TEST_GENERAL_PATH = DATA_DIR / "test_original.csv"
TEST_KNN_PATH = DATA_DIR / "test_original_knn_raw.csv"

TUNING_RESULTS_PATH = MODELS_DIR / "tuning_results.csv"
ENSEMBLE_RESULTS_PATH = MODELS_DIR / "ensemble_results.csv"
BASELINE_SUMMARY_PATH = MODELS_DIR / "model_comparison_summary.csv"
EXPERIMENT_LOG_PATH = MODELS_DIR / "experiments_log.csv"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
candidate_rows = []

if TUNING_RESULTS_PATH.exists():
    tuning_results_df = pd.read_csv(TUNING_RESULTS_PATH)
    temp = tuning_results_df.copy()
    temp["candidate_label"] = temp["model_key"].apply(lambda x: f"tuned_{x}")
    temp["candidate_group"] = "tuned"
    temp["selection_metric_cv"] = temp["best_cv_score"]
    candidate_rows.append(temp)

if ENSEMBLE_RESULTS_PATH.exists():
    ensemble_results_df = pd.read_csv(ENSEMBLE_RESULTS_PATH)
    temp = ensemble_results_df.copy()
    temp["candidate_label"] = temp["model_key"]
    temp["candidate_group"] = temp["candidate_type"]
    temp["selection_metric_cv"] = temp["test_precision_mean"]
    candidate_rows.append(temp)

candidate_pool_df = pd.concat(candidate_rows, ignore_index=True, sort=False)
candidate_pool_df = candidate_pool_df.sort_values(by="selection_metric_cv", ascending=False).reset_index(drop=True)
display(candidate_pool_df[["candidate_label", "candidate_group", "selection_metric_cv", "artifact_path", "data_variant"]])

In [ ]:
if FINAL_MODEL_KEY is None:
    print("⚠️ Define FINAL_MODEL_KEY antes de evaluar en test set.")
if not CONFIRM_FINAL_TEST_EVALUATION:
    print("⚠️ CONFIRM_FINAL_TEST_EVALUATION sigue en False.")

In [ ]:
test_general_df = pd.read_csv(TEST_GENERAL_PATH)
test_knn_df = pd.read_csv(TEST_KNN_PATH) if TEST_KNN_PATH.exists() else test_general_df.copy()

test_datasets = {"general": test_general_df, "knn_raw": test_knn_df}

def get_test_xy(data_variant: str):
    df = test_datasets[data_variant].copy()
    X = df.drop(columns=[TARGET_COL]).copy()
    y = df[TARGET_COL].copy()
    return X, y

In [ ]:
if FINAL_MODEL_KEY is None or not CONFIRM_FINAL_TEST_EVALUATION:
    raise RuntimeError("Configura FINAL_MODEL_KEY y pon CONFIRM_FINAL_TEST_EVALUATION=True antes de ejecutar la evaluación final.")

selected_row = candidate_pool_df.loc[candidate_pool_df["candidate_label"] == FINAL_MODEL_KEY]
if selected_row.empty:
    raise KeyError(f"No se encontró FINAL_MODEL_KEY={FINAL_MODEL_KEY} en el pool de candidatos.")

selected_row = selected_row.iloc[0]
final_model_path = Path(selected_row["artifact_path"])
final_data_variant = selected_row.get("data_variant", "general")
final_model = joblib.load(final_model_path)

X_test, y_test = get_test_xy(final_data_variant)
y_pred = final_model.predict(X_test)

if hasattr(final_model, "predict_proba"):
    y_score = final_model.predict_proba(X_test)[:, 1]
elif hasattr(final_model, "decision_function"):
    y_score = final_model.decision_function(X_test)
else:
    y_score = None

final_metrics = {
    "final_model_key": FINAL_MODEL_KEY,
    "candidate_group": selected_row["candidate_group"],
    "data_variant": final_data_variant,
    "test_accuracy": accuracy_score(y_test, y_pred),
    "test_precision": precision_score(y_test, y_pred, zero_division=0),
    "test_recall": recall_score(y_test, y_pred, zero_division=0),
    "test_f1": f1_score(y_test, y_pred, zero_division=0),
    "test_roc_auc": roc_auc_score(y_test, y_score) if y_score is not None else np.nan,
}

final_metrics_df = pd.DataFrame([final_metrics])
display(final_metrics_df)

final_classification_report_df = pd.DataFrame(
    classification_report(y_test, y_pred, output_dict=True, zero_division=0)
).T.reset_index().rename(columns={"index": "label"})
display(final_classification_report_df)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax, colorbar=False)
ax.set_title(f"Confusion Matrix — {FINAL_MODEL_KEY}")
plt.tight_layout()

final_cm_path = REPORTS_DIR / "final_confusion_matrix.png"
plt.savefig(final_cm_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", final_cm_path)

In [ ]:
def bootstrap_metric(y_true, y_pred_or_score, metric_fn, n=1000, seed=42):
    rng = np.random.default_rng(seed)
    scores = []
    y_true = np.asarray(y_true)
    y_pred_or_score = np.asarray(y_pred_or_score)

    for _ in range(n):
        idx = rng.choice(len(y_true), size=len(y_true), replace=True)
        try:
            score = metric_fn(y_true[idx], y_pred_or_score[idx])
            scores.append(score)
        except Exception:
            continue

    if not scores:
        return (np.nan, np.nan)
    return tuple(np.percentile(scores, [2.5, 97.5]))

bootstrap_rows = []
for metric_name, metric_fn, values in [
    ("precision", precision_score, y_pred),
    ("recall", recall_score, y_pred),
    ("f1", f1_score, y_pred),
    ("accuracy", accuracy_score, y_pred),
]:
    low, high = bootstrap_metric(y_test, values, lambda yt, yp: metric_fn(yt, yp, zero_division=0) if metric_name != "accuracy" else metric_fn(yt, yp))
    bootstrap_rows.append({"metric": metric_name, "ci_low": low, "ci_high": high})

if y_score is not None:
    low, high = bootstrap_metric(y_test, y_score, roc_auc_score)
    bootstrap_rows.append({"metric": "roc_auc", "ci_low": low, "ci_high": high})

bootstrap_df = pd.DataFrame(bootstrap_rows)
display(bootstrap_df)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(
    x=bootstrap_df["metric"],
    y=(bootstrap_df["ci_low"] + bootstrap_df["ci_high"]) / 2,
    yerr=(bootstrap_df["ci_high"] - bootstrap_df["ci_low"]) / 2,
    fmt="o",
)
ax.set_title("Bootstrap 95% CI — Final Model")
ax.set_ylabel("Metric value")
plt.tight_layout()

bootstrap_fig_path = REPORTS_DIR / "final_bootstrap_intervals.png"
plt.savefig(bootstrap_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved:", bootstrap_fig_path)

In [ ]:
baseline_vs_final_df = pd.DataFrame()
if BASELINE_SUMMARY_PATH.exists():
    baseline_summary_df = pd.read_csv(BASELINE_SUMMARY_PATH)
    baseline_best = baseline_summary_df.sort_values(by="test_precision_mean", ascending=False).iloc[0]

    baseline_vs_final_df = pd.DataFrame([
        {"metric": "precision", "baseline_sprint3": baseline_best.get("test_precision_mean", np.nan), "final_sprint4": final_metrics["test_precision"]},
        {"metric": "recall", "baseline_sprint3": baseline_best.get("test_recall_mean", np.nan), "final_sprint4": final_metrics["test_recall"]},
        {"metric": "f1", "baseline_sprint3": baseline_best.get("test_f1_mean", np.nan), "final_sprint4": final_metrics["test_f1"]},
        {"metric": "roc_auc", "baseline_sprint3": baseline_best.get("test_roc_auc_mean", np.nan), "final_sprint4": final_metrics["test_roc_auc"]},
        {"metric": "accuracy", "baseline_sprint3": baseline_best.get("test_accuracy_mean", np.nan), "final_sprint4": final_metrics["test_accuracy"]},
    ])
    baseline_vs_final_df["improvement_pct"] = np.where(
        baseline_vs_final_df["baseline_sprint3"] != 0,
        (baseline_vs_final_df["final_sprint4"] - baseline_vs_final_df["baseline_sprint3"]) / baseline_vs_final_df["baseline_sprint3"] * 100,
        np.nan,
    )

display(baseline_vs_final_df)

In [ ]:
FINAL_MODEL_PATH = MODELS_DIR / "final_model.pkl"
FINAL_METRICS_PATH = MODELS_DIR / "final_validation_metrics.csv"
BASELINE_VS_FINAL_PATH = MODELS_DIR / "baseline_vs_final_comparison.csv"

joblib.dump(final_model, FINAL_MODEL_PATH)
final_metrics_df.to_csv(FINAL_METRICS_PATH, index=False)
if not baseline_vs_final_df.empty:
    baseline_vs_final_df.to_csv(BASELINE_VS_FINAL_PATH, index=False)

print("Saved:", FINAL_MODEL_PATH)
print("Saved:", FINAL_METRICS_PATH)
print("Saved:", BASELINE_VS_FINAL_PATH if not baseline_vs_final_df.empty else "comparison skipped")

In [ ]:
experiments_log_df = pd.read_csv(EXPERIMENT_LOG_PATH) if EXPERIMENT_LOG_PATH.exists() else pd.DataFrame()

final_log_row = {
    "log_timestamp": pd.Timestamp.now().isoformat(),
    "sprint": 4,
    "stage": "final_validation",
    "model_key": FINAL_MODEL_KEY,
    "status": "final_selected",
    "primary_metric": PRIMARY_METRIC,
    "artifact_path": str(FINAL_MODEL_PATH),
    "test_accuracy": final_metrics["test_accuracy"],
    "test_precision": final_metrics["test_precision"],
    "test_recall": final_metrics["test_recall"],
    "test_f1": final_metrics["test_f1"],
    "test_roc_auc": final_metrics["test_roc_auc"],
    "notes": "Final model validated on test set (used once).",
}

experiments_log_df = pd.concat([experiments_log_df, pd.DataFrame([final_log_row])], ignore_index=True)
experiments_log_df.to_csv(EXPERIMENT_LOG_PATH, index=False)
print("Updated:", EXPERIMENT_LOG_PATH)